In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"

for file in DATA_DIR.glob("*.csv"):
    name = file.stem

    if "nav_history" in name:
        nav = pd.read_csv(file)
        print("NAV columns:", nav.columns.tolist())

    elif "benchmark_indices" in name:
        benchmarks = pd.read_csv(file)
        print("Benchmark columns:", benchmarks.columns.tolist())

Benchmark columns: ['date', 'index_name', 'close_value']
NAV columns: ['amfi_code', 'date', 'nav']


In [2]:
# Convert dates and sort NAV data
nav["date"] = pd.to_datetime(nav["date"])
nav = nav.sort_values(["amfi_code", "date"])

# Calculate daily returns for each mutual fund
nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"].pct_change()
)

# Calculate annualized returns using 252 trading days
annual_returns = (
    nav.groupby("amfi_code")["daily_return"]
    .apply(
        lambda x: (
            (1 + x.dropna()).prod()
            ** (252 / len(x.dropna())) - 1
        ) * 100
        if len(x.dropna()) > 0 else np.nan
    )
    .reset_index(name="annualized_return_pct")
)

display(annual_returns.head(10))
print("Total schemes:", annual_returns["amfi_code"].nunique())
print("Missing annualized returns:", annual_returns["annualized_return_pct"].isna().sum())

,amfi_code,annualized_return_pct
0,100016,2.543487
1,100025,4.298652
2,100033,28.927866
3,101206,22.626535
4,101207,7.650159
5,101208,6.273897
6,102885,17.540419
7,102886,1.130448
8,102887,16.205501
9,118632,23.116077


Total schemes: 40
Missing annualized returns: 0


In [3]:
# Load fund master data
fund_file = next(DATA_DIR.glob("*fund_master*.csv"))
funds = pd.read_csv(fund_file)

print("Fund Master Columns:")
print(funds.columns.tolist())

print("\nFund Benchmark Mapping:")
display(funds[["amfi_code", "scheme_name", "benchmark"]].head(10))


Fund Master Columns:
['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']

Fund Benchmark Mapping:


,amfi_code,scheme_name,benchmark
0,119551,SBI Bluechip Fund - Regular Plan - Growth,NIFTY 100 TRI
1,119552,SBI Bluechip Fund - Direct Plan - Growth,NIFTY 100 TRI
2,119598,SBI Small Cap Fund - Regular Plan - Growth,BSE 250 SmallCap TRI
3,119599,SBI Small Cap Fund - Direct Plan - Growth,BSE 250 SmallCap TRI
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,CRISIL Dynamic Gilt Index
5,100016,HDFC Top 100 Fund - Regular Plan - Growth,NIFTY 100 TRI
6,125497,HDFC Top 100 Fund - Direct Plan - Growth,NIFTY 100 TRI
7,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,NIFTY Midcap 150 TRI
8,125498,HDFC Mid-Cap Opportunities Fund - Direct - Growth,NIFTY Midcap 150 TRI
9,100025,HDFC Short Term Debt Fund - Regular - Growth,CRISIL Short Term Bond Index


In [4]:
# Prepare NAV returns
nav["date"] = pd.to_datetime(nav["date"])
nav = nav.sort_values(["amfi_code", "date"])

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"].pct_change()
)

# Prepare benchmark returns
benchmarks["date"] = pd.to_datetime(benchmarks["date"])
benchmarks = benchmarks.sort_values(["index_name", "date"])

benchmarks["benchmark_return"] = (
    benchmarks.groupby("index_name")["close_value"].pct_change()
)

# Merge fund returns with fund benchmark mapping
fund_returns = nav.merge(
    funds[["amfi_code", "scheme_name", "benchmark", "category"]],
    on="amfi_code",
    how="left"
)

merged = fund_returns.merge(
    benchmarks[["date", "index_name", "benchmark_return"]],
    left_on=["date", "benchmark"],
    right_on=["date", "index_name"],
    how="inner"
)

print("Matched fund-benchmark daily return rows:", len(merged))
print("Funds with matched benchmark data:", merged["amfi_code"].nunique())
print("Benchmark names available:")
print(sorted(benchmarks["index_name"].dropna().unique()))

Matched fund-benchmark daily return rows: 0
Funds with matched benchmark data: 0
Benchmark names available:
['BSE_SMALLCAP', 'CRISIL_GILT', 'CRISIL_LIQUID', 'NIFTY100', 'NIFTY50', 'NIFTY500', 'NIFTY_MIDCAP150']


In [5]:
# Map fund benchmark names to names in benchmark data
benchmark_mapping = {
    "NIFTY 100 TRI": "NIFTY100",
    "BSE 250 SmallCap TRI": "BSE_SMALLCAP",
    "CRISIL Dynamic Gilt Index": "CRISIL_GILT",
    "CRISIL Short Term Bond Index": "CRISIL_GILT",
    "NIFTY Midcap 150 TRI": "NIFTY_MIDCAP150",
    "NIFTY 50 TRI": "NIFTY50",
    "NIFTY 500 TRI": "NIFTY500",
}

funds["benchmark_key"] = funds["benchmark"].map(benchmark_mapping)

print("Fund benchmark names:")
print(funds[["benchmark", "benchmark_key"]].drop_duplicates().to_string(index=False))

print("\nUnmapped benchmark names:")
print(funds.loc[funds["benchmark_key"].isna(), "benchmark"].drop_duplicates().tolist())

Fund benchmark names:
                   benchmark   benchmark_key
               NIFTY 100 TRI        NIFTY100
        BSE 250 SmallCap TRI    BSE_SMALLCAP
   CRISIL Dynamic Gilt Index     CRISIL_GILT
        NIFTY Midcap 150 TRI NIFTY_MIDCAP150
CRISIL Short Term Bond Index     CRISIL_GILT
               NIFTY 500 TRI        NIFTY500
 CRISIL Liquid Fund AI Index             NaN
                NIFTY 50 TRI         NIFTY50
         NIFTY Midcap 50 TRI             NaN
  NIFTY Large Midcap 250 TRI             NaN

Unmapped benchmark names:
['CRISIL Liquid Fund AI Index', 'NIFTY Midcap 50 TRI', 'NIFTY Large Midcap 250 TRI']


In [6]:
# Correct benchmark mapping
benchmark_mapping = {
    "NIFTY 100 TRI": "NIFTY100",
    "BSE 250 SmallCap TRI": "BSE_SMALLCAP",
    "CRISIL Dynamic Gilt Index": "CRISIL_GILT",
    "NIFTY Midcap 150 TRI": "NIFTY_MIDCAP150",
    "NIFTY 500 TRI": "NIFTY500",
    "NIFTY 50 TRI": "NIFTY50",
}

funds["benchmark_key"] = funds["benchmark"].map(benchmark_mapping)

print("Mapped schemes:", funds["benchmark_key"].notna().sum())
print("Unmapped schemes:", funds["benchmark_key"].isna().sum())

display(
    funds.loc[
        funds["benchmark_key"].isna(),
        ["scheme_name", "benchmark"]
    ]
)

Mapped schemes: 34
Unmapped schemes: 6


,scheme_name,benchmark
9,HDFC Short Term Debt Fund - Regular - Growth,CRISIL Short Term Bond Index
14,ICICI Pru Liquid Fund - Regular - Growth,CRISIL Liquid Fund AI Index
23,Kotak Liquid Fund - Regular - Growth,CRISIL Liquid Fund AI Index
26,Axis Midcap Fund - Regular - Growth,NIFTY Midcap 50 TRI
30,ABSL Liquid Fund - Regular - Growth,CRISIL Liquid Fund AI Index
35,Mirae Asset Emerging Bluechip Fund - Regular -...,NIFTY Large Midcap 250 TRI


In [7]:
# Rebuild fund returns with corrected benchmark mapping
fund_returns = nav.merge(
    funds[["amfi_code", "scheme_name", "benchmark", "benchmark_key", "category"]],
    on="amfi_code",
    how="left"
)

merged = fund_returns.merge(
    benchmarks[["date", "index_name", "benchmark_return"]],
    left_on=["date", "benchmark_key"],
    right_on=["date", "index_name"],
    how="inner"
)

print("Matched daily return rows:", len(merged))
print("Schemes with matched benchmark data:", merged["amfi_code"].nunique())
print("Schemes still unmatched:", 40 - merged["amfi_code"].nunique())

display(
    merged[["scheme_name", "date", "daily_return", "benchmark_return"]].head()
)

Matched daily return rows: 39100
Schemes with matched benchmark data: 34
Schemes still unmatched: 6


,scheme_name,date,daily_return,benchmark_return
0,HDFC Top 100 Fund - Regular Plan - Growth,2022-01-03,NaN,NaN
1,HDFC Top 100 Fund - Regular Plan - Growth,2022-01-04,-0.010306,-0.013540
2,HDFC Top 100 Fund - Regular Plan - Growth,2022-01-05,0.012865,0.004003
3,HDFC Top 100 Fund - Regular Plan - Growth,2022-01-06,-0.011377,-0.002935
4,HDFC Top 100 Fund - Regular Plan - Growth,2022-01-07,-0.001210,0.006150


In [8]:
# Calculate Sharpe Ratio, Beta and 95% VaR

RISK_FREE_RATE = 0.06  # Assumed annual risk-free rate: 6%
TRADING_DAYS = 252

def calculate_metrics(group):
    valid = group[["daily_return", "benchmark_return"]].dropna()

    if len(valid) < 2:
        return pd.Series({
            "observations": len(valid),
            "annualized_return_pct": np.nan,
            "annualized_volatility_pct": np.nan,
            "sharpe_ratio": np.nan,
            "beta": np.nan,
            "var_95_daily_pct": np.nan
        })

    fund_r = valid["daily_return"]
    bench_r = valid["benchmark_return"]

    daily_rf = (1 + RISK_FREE_RATE) ** (1 / TRADING_DAYS) - 1

    volatility = fund_r.std(ddof=1) * np.sqrt(TRADING_DAYS)

    sharpe = (
        ((fund_r.mean() - daily_rf) * TRADING_DAYS)
        / (fund_r.std(ddof=1) * np.sqrt(TRADING_DAYS))
        if fund_r.std(ddof=1) > 0 else np.nan
    )

    beta = (
        fund_r.cov(bench_r) / bench_r.var(ddof=1)
        if bench_r.var(ddof=1) > 0 else np.nan
    )

    annualized_return = (
        ((1 + fund_r).prod() ** (TRADING_DAYS / len(fund_r)) - 1) * 100
    )

    return pd.Series({
        "observations": len(valid),
        "annualized_return_pct": annualized_return,
        "annualized_volatility_pct": volatility * 100,
        "sharpe_ratio": sharpe,
        "beta": beta,
        "var_95_daily_pct": -fund_r.quantile(0.05) * 100
    })


metrics = (
    merged.groupby(["amfi_code", "scheme_name", "benchmark"])
    .apply(calculate_metrics, include_groups=False)
    .reset_index()
)

display(metrics.head(10))
print("Schemes analyzed:", metrics["amfi_code"].nunique())
print("Missing Beta:", metrics["beta"].isna().sum())

,amfi_code,scheme_name,benchmark,observations,annualized_return_pct,annualized_volatility_pct,sharpe_ratio,beta,var_95_daily_pct
0,100016,HDFC Top 100 Fund - Regular Plan - Growth,NIFTY 100 TRI,1149.0,2.543487,14.548135,-0.155296,-0.058268,1.436364
1,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,NIFTY Midcap 150 TRI,1149.0,28.927866,18.936711,1.129208,0.031026,1.903354
2,101206,ABSL Frontline Equity Fund - Regular - Growth,NIFTY 100 TRI,1149.0,22.626535,14.568213,1.073371,0.021086,1.328166
3,101207,ABSL Small Cap Fund - Regular - Growth,BSE 250 SmallCap TRI,1149.0,7.650159,25.797322,0.188727,-0.048604,2.602125
4,102885,UTI Nifty 50 Index Fund - Regular - Growth,NIFTY 50 TRI,1149.0,17.540419,12.836628,0.869483,-0.001884,1.261263
5,102886,UTI Mid Cap Fund - Regular - Growth,NIFTY Midcap 150 TRI,1149.0,1.130448,18.134576,-0.168751,-0.049090,1.922028
6,102887,UTI Flexi Cap Fund - Regular - Growth,NIFTY 500 TRI,1149.0,16.205501,15.758367,0.662189,-0.014987,1.523162
7,118632,Nippon India Large Cap Fund - Regular - Growth,NIFTY 100 TRI,1149.0,23.116077,14.148352,1.129187,-0.008354,1.395405
8,118633,Nippon India Large Cap Fund - Direct - Growth,NIFTY 100 TRI,1149.0,15.661395,13.997214,0.693248,-0.036639,1.344253
9,118634,Nippon India Small Cap Fund - Regular - Growth,BSE 250 SmallCap TRI,1149.0,15.756667,25.241521,0.475074,0.000054,2.543811


Schemes analyzed: 34
Missing Beta: 0


In [9]:
# Save performance metrics
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

output_file = REPORTS_DIR / "fund_performance_metrics.csv"

metrics.to_csv(output_file, index=False)

print("Performance metrics saved successfully!")
print("File location:", output_file)
print("Total schemes saved:", len(metrics))

display(metrics.head())

Performance metrics saved successfully!
File location: C:\Users\asus\mutual-funds-analysis\reports\fund_performance_metrics.csv
Total schemes saved: 34


,amfi_code,scheme_name,benchmark,observations,annualized_return_pct,annualized_volatility_pct,sharpe_ratio,beta,var_95_daily_pct
0,100016,HDFC Top 100 Fund - Regular Plan - Growth,NIFTY 100 TRI,1149.0,2.543487,14.548135,-0.155296,-0.058268,1.436364
1,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,NIFTY Midcap 150 TRI,1149.0,28.927866,18.936711,1.129208,0.031026,1.903354
2,101206,ABSL Frontline Equity Fund - Regular - Growth,NIFTY 100 TRI,1149.0,22.626535,14.568213,1.073371,0.021086,1.328166
3,101207,ABSL Small Cap Fund - Regular - Growth,BSE 250 SmallCap TRI,1149.0,7.650159,25.797322,0.188727,-0.048604,2.602125
4,102885,UTI Nifty 50 Index Fund - Regular - Growth,NIFTY 50 TRI,1149.0,17.540419,12.836628,0.869483,-0.001884,1.261263


In [10]:
# Validate performance metric ranges

print("Annualized Return (%):")
print(metrics["annualized_return_pct"].describe())

print("\nAnnualized Volatility (%):")
print(metrics["annualized_volatility_pct"].describe())

print("\nSharpe Ratio:")
print(metrics["sharpe_ratio"].describe())

print("\nBeta:")
print(metrics["beta"].describe())

print("\nDaily VaR 95 (%):")
print(metrics["var_95_daily_pct"].describe())

print("\nCheck for negative VaR values:")
print(metrics.loc[
    metrics["var_95_daily_pct"] < 0,
    ["scheme_name", "var_95_daily_pct"]
].head())

Annualized Return (%):
count    34.000000
mean     16.782735
std       9.910473
min       1.130448
25%       7.620416
50%      16.322459
75%      24.376468
max      31.512438
Name: annualized_return_pct, dtype: float64

Annualized Volatility (%):
count    34.000000
mean     16.397625
std       5.217868
min       3.967062
25%      13.958820
50%      14.618540
75%      18.297967
max      25.797322
Name: annualized_volatility_pct, dtype: float64

Sharpe Ratio:
count    34.000000
mean      0.652826
std       0.507739
min      -0.187911
25%       0.180459
50%       0.704949
75%       1.073363
max       1.495667
Name: sharpe_ratio, dtype: float64

Beta:
count    34.000000
mean     -0.003108
std       0.032584
min      -0.058268
25%      -0.030353
50%      -0.002967
75%       0.023034
max       0.059234
Name: beta, dtype: float64

Daily VaR 95 (%):
count    34.000000
mean      1.619628
std       0.544761
min       0.380415
25%       1.351323
50%       1.443116
75%       1.876017
max       2.6

In [11]:
# Check benchmark return statistics
print("Benchmark return summary:")
print(
    benchmarks.groupby("index_name")["benchmark_return"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(6)
)

print("\nFund daily return summary:")
print(
    nav.groupby("amfi_code")["daily_return"]
    .agg(["count", "mean", "std", "min", "max"])
    .describe()
    .round(6)
)

Benchmark return summary:
                 count      mean       std       min       max
index_name                                                    
BSE_SMALLCAP      1149  0.001039  0.013948 -0.039716  0.041854
CRISIL_GILT       1149  0.000405  0.002602 -0.007441  0.008212
CRISIL_LIQUID     1149  0.000252  0.000326 -0.000780  0.001261
NIFTY100          1149  0.000122  0.008119 -0.026873  0.026795
NIFTY50           1149  0.000082  0.008106 -0.023769  0.023067
NIFTY500          1149  0.000850  0.008734 -0.028623  0.031022
NIFTY_MIDCAP150   1149  0.001121  0.011229 -0.035748  0.035232

Fund daily return summary:
        count       mean        std        min        max
count    40.0  40.000000  40.000000  40.000000  40.000000
mean   1149.0   0.000631   0.009414  -0.029884   0.032574
std       0.0   0.000348   0.004205   0.013980   0.014814
min    1149.0   0.000110   0.000311  -0.058102   0.001197
25%    1149.0   0.000273   0.008724  -0.038894   0.028536
50%    1149.0   0.000648   0.00

In [12]:
# Manually verify Beta for one fund
sample = merged[
    merged["amfi_code"] == merged["amfi_code"].iloc[0]
][["daily_return", "benchmark_return"]].dropna()

manual_beta = (
    sample["daily_return"].cov(sample["benchmark_return"])
    / sample["benchmark_return"].var()
)

print("Fund:", merged["scheme_name"].iloc[0])
print("Benchmark:", merged["benchmark"].iloc[0])
print("Matched observations:", len(sample))
print("Fund daily volatility:", sample["daily_return"].std())
print("Benchmark daily volatility:", sample["benchmark_return"].std())
print("Manually calculated Beta:", manual_beta)
print("Beta saved in metrics:", metrics.iloc[0]["beta"])

Fund: HDFC Top 100 Fund - Regular Plan - Growth
Benchmark: NIFTY 100 TRI
Matched observations: 1149
Fund daily volatility: 0.009164463506291045
Benchmark daily volatility: 0.008119419678050336
Manually calculated Beta: -0.05826843143880895
Beta saved in metrics: -0.05826843143880895


In [13]:
print("Correlation:", sample["daily_return"].corr(sample["benchmark_return"]))

print("\nFund return statistics:")
print(sample["daily_return"].describe().round(6))

print("\nBenchmark return statistics:")
print(sample["benchmark_return"].describe().round(6))

Correlation: -0.05162395469288775

Fund return statistics:
count    1149.000000
mean        0.000142
std         0.009164
min        -0.024744
25%        -0.006282
50%        -0.000070
75%         0.006523
max         0.032145
Name: daily_return, dtype: float64

Benchmark return statistics:
count    1149.000000
mean        0.000122
std         0.008119
min        -0.026873
25%        -0.005385
50%        -0.000112
75%         0.005456
max         0.026795
Name: benchmark_return, dtype: float64


In [14]:
correlation_check = (
    merged.groupby(["amfi_code", "scheme_name", "benchmark"])
    .apply(
        lambda g: g["daily_return"].corr(g["benchmark_return"]),
        include_groups=False
    )
    .reset_index(name="correlation")
)

display(
    correlation_check.sort_values("correlation").round(4)
)

print("\nCorrelation summary:")
print(correlation_check["correlation"].describe().round(4))

,amfi_code,scheme_name,benchmark,correlation
0,100016,HDFC Top 100 Fund - Regular Plan - Growth,NIFTY 100 TRI,-0.0516
5,102886,UTI Mid Cap Fund - Regular - Growth,NIFTY Midcap 150 TRI,-0.0483
18,119598,SBI Small Cap Fund - Regular Plan - Growth,BSE 250 SmallCap TRI,-0.0419
3,101207,ABSL Small Cap Fund - Regular - Growth,BSE 250 SmallCap TRI,-0.0417
14,119095,Axis Small Cap Fund - Regular - Growth,BSE 250 SmallCap TRI,-0.0403
20,120503,ICICI Pru Bluechip Fund - Regular - Growth,NIFTY 100 TRI,-0.0374
25,120842,Kotak Emerging Equity Fund - Regular - Growth,NIFTY Midcap 150 TRI,-0.0349
8,118633,Nippon India Large Cap Fund - Direct - Growth,NIFTY 100 TRI,-0.0337
16,119551,SBI Bluechip Fund - Regular Plan - Growth,NIFTY 100 TRI,-0.0298
17,119552,SBI Bluechip Fund - Direct Plan - Growth,NIFTY 100 TRI,-0.0242



Correlation summary:
count    34.0000
mean     -0.0030
std       0.0297
min      -0.0516
25%      -0.0284
50%      -0.0028
75%       0.0208
max       0.0575
Name: correlation, dtype: float64


In [15]:
# Inspect the index fund against its benchmark
fund_code = 102885  # UTI Nifty 50 Index Fund

check = merged[merged["amfi_code"] == fund_code].copy()

print("Fund:", check["scheme_name"].iloc[0])
print("Benchmark:", check["benchmark"].iloc[0])
print("Rows:", len(check))

print("\nFirst 10 matched daily returns:")
display(
    check[
        ["date", "daily_return", "benchmark_return"]
    ].head(10).round(6)
)

print("\nReturn correlation:")
print(check["daily_return"].corr(check["benchmark_return"]))

print("\nSame-day return difference:")
print(
    (check["daily_return"] - check["benchmark_return"])
    .describe()
    .round(6)
)

Fund: UTI Nifty 50 Index Fund - Regular - Growth
Benchmark: NIFTY 50 TRI
Rows: 1150

First 10 matched daily returns:


,date,daily_return,benchmark_return
4600,2022-01-03,NaN,NaN
4601,2022-01-04,0.011122,0.011253
4602,2022-01-05,-0.007878,0.008220
4603,2022-01-06,0.015277,0.002437
4604,2022-01-07,-0.009369,-0.006676
4605,2022-01-10,-0.001202,0.020592
4606,2022-01-11,0.011263,0.007289
4607,2022-01-12,0.004676,-0.005172
4608,2022-01-13,-0.009585,0.000909
4609,2022-01-14,0.009862,-0.011403



Return correlation:
-0.0018886842292548819

Same-day return difference:
count    1149.000000
mean        0.000592
std         0.011460
min        -0.035692
25%        -0.007426
50%         0.000397
75%         0.008674
max         0.037028
dtype: float64


In [16]:
# Compare cumulative fund and benchmark performance
check = check.sort_values("date").copy()
valid = check.dropna(subset=["daily_return", "benchmark_return"]).copy()

valid["fund_growth"] = (1 + valid["daily_return"]).cumprod()
valid["benchmark_growth"] = (1 + valid["benchmark_return"]).cumprod()

print("Fund cumulative growth:", round(valid["fund_growth"].iloc[-1], 4))
print("Benchmark cumulative growth:", round(valid["benchmark_growth"].iloc[-1], 4))

print("\nFirst and last NAV values:")
fund_nav = nav[nav["amfi_code"] == fund_code].sort_values("date")
display(fund_nav.iloc[[0, -1]][["date", "nav"]])

print("\nFund and benchmark daily returns correlation:")
print(valid["daily_return"].corr(valid["benchmark_return"]))

Fund cumulative growth: 2.0894
Benchmark cumulative growth: 1.0586

First and last NAV values:


,date,nav
35650,2022-01-03,89.8728
36799,2026-05-29,187.7797



Fund and benchmark daily returns correlation:
-0.0018886842292548819


In [17]:
# Find the UTI Nifty 50 Index Fund in scheme performance data
performance_file = next(DATA_DIR.glob("*scheme_performance*.csv"))
performance_data = pd.read_csv(performance_file)

print("Available columns:")
print(performance_data.columns.tolist())

display(
    performance_data[
        performance_data.astype(str)
        .apply(
            lambda col: col.str.contains(
                "UTI Nifty 50 Index Fund",
                case=False,
                na=False
            )
        )
        .any(axis=1)
    ]
)

Available columns:
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
31,102885,UTI Nifty 50 Index Fund - Regular - Growth,UTI Mutual Fund,Index,Regular,13.76,12.1,11.31,11.17,0.93,0.9,0.93,1.29,13.0,-24.42,7350,1.57,4,Moderate


In [18]:
# Compare NAV-derived returns with reported performance
fund_code = 102885

fund_nav = (
    nav[nav["amfi_code"] == fund_code]
    .sort_values("date")
    .copy()
)

fund_nav["date"] = pd.to_datetime(fund_nav["date"])

latest_date = fund_nav["date"].max()
print("Latest NAV date:", latest_date.date())

for years in [1, 3]:
    target_date = latest_date - pd.DateOffset(years=years)
    earlier = fund_nav[fund_nav["date"] <= target_date]

    if not earlier.empty:
        start_row = earlier.iloc[-1]
        end_row = fund_nav.iloc[-1]

        total_return = (
            end_row["nav"] / start_row["nav"] - 1
        ) * 100

        print(f"\n{years}-year NAV return: {total_return:.2f}%")
        print("Start date:", start_row["date"].date())
        print("Start NAV:", start_row["nav"])
        print("End NAV:", end_row["nav"])

Latest NAV date: 2026-05-29

1-year NAV return: 20.21%
Start date: 2025-05-29
Start NAV: 156.2127
End NAV: 187.7797

3-year NAV return: 71.37%
Start date: 2023-05-29
Start NAV: 109.5778
End NAV: 187.7797


In [19]:
# Check unusually large daily NAV movements
fund_nav = fund_nav.sort_values("date").copy()
fund_nav["daily_return_pct"] = fund_nav["nav"].pct_change() * 100

print("Daily NAV return summary:")
print(fund_nav["daily_return_pct"].describe().round(2))

print("\nLargest positive daily returns:")
display(
    fund_nav.nlargest(10, "daily_return_pct")[
        ["date", "nav", "daily_return_pct"]
    ].round(4)
)

print("\nLargest negative daily returns:")
display(
    fund_nav.nsmallest(10, "daily_return_pct")[
        ["date", "nav", "daily_return_pct"]
    ].round(4)
)

Daily NAV return summary:
count    1149.00
mean        0.07
std         0.81
min        -2.11
25%        -0.49
50%         0.06
75%         0.61
max         2.56
Name: daily_return_pct, dtype: float64

Largest positive daily returns:


,date,nav,daily_return_pct
35968,2023-03-23,106.2749,2.5565
36769,2026-04-17,180.6158,2.4117
36431,2024-12-31,137.6930,2.3996
36221,2024-03-12,131.6703,2.3150
36511,2025-04-22,158.3770,2.2828
36593,2025-08-14,160.0956,2.2217
36394,2024-11-08,132.6091,2.1514
36326,2024-08-06,123.3393,2.1033
36408,2024-11-28,130.4519,2.0916
35789,2022-07-15,93.6061,2.0758



Largest negative daily returns:


,date,nav,daily_return_pct
35687,2022-02-23,90.4800,-2.1104
35693,2022-03-03,86.4521,-2.0632
35924,2023-01-20,100.6154,-1.9707
35703,2022-03-17,85.5917,-1.9578
36217,2024-03-06,128.7858,-1.9381
35846,2022-10-04,90.7575,-1.9360
36056,2023-07-25,111.0733,-1.9090
36401,2024-11-19,127.4958,-1.8794
36475,2025-03-03,140.5004,-1.8598
35724,2022-04-15,87.0724,-1.8456


In [20]:
from pathlib import Path

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

output_file = REPORTS_DIR / "fund_performance_metrics.csv"
metrics.to_csv(output_file, index=False)

print("Performance metrics saved successfully.")
print("File:", output_file)
print("Number of funds:", len(metrics))

Performance metrics saved successfully.
File: C:\Users\asus\mutual-funds-analysis\reports\fund_performance_metrics.csv
Number of funds: 34
